# 01 – Data Check

Verify that all required OIFS output files exist and contain the
variables needed for track and core-size analysis.

**Required variables** (from Ribberink et al. 2025 functions):
- `msl` (MSLP, Pa) from `ICMGGac3u+*` → storm tracking
- `10u`, `10v` (10 m winds, m s⁻¹) from `ICMGGac3u+*` → wind radius
- `z` (geopotential, m² s⁻²) at 300, 600, 900 hPa from `ICMSHac3u+*` → CPS
- IBTrACS best track for Kirk 2024 → track initialisation

In [ ]:
import sys, os
# Add scripts and ribberink_code to path
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import xarray as xr
import numpy as np
import pandas as pd
import glob

from oifs_adapter import RUNS, OIFSRun, INIT_DATETIME

## 1. Check run directories and file counts

In [ ]:
for run_name, run_dir in RUNS.items():
    gg_files = sorted(glob.glob(os.path.join(run_dir, 'ICMGGac3u+??????')))
    sh_files = sorted(glob.glob(os.path.join(run_dir, 'ICMSHac3u+??????')))
    ua_files = sorted(glob.glob(os.path.join(run_dir, 'ICMUAac3u+??????')))

    if not gg_files:
        print(f'{run_name}: ⚠️  DIRECTORY NOT FOUND or EMPTY')
        continue

    max_step_gg = int(gg_files[-1].split('+')[-1]) if gg_files else 0
    max_step_sh = int(sh_files[-1].split('+')[-1]) if sh_files else 0

    print(f"Run: {run_name}")
    print(f"  ICMGG files: {len(gg_files):3d} (max step: {max_step_gg} h = {max_step_gg/24:.1f} days)")
    print(f"  ICMSH files: {len(sh_files):3d} (max step: {max_step_sh} h)")
    print(f"  ICMUA files: {len(ua_files):3d}")
    print()

## 2. Inspect variables in the first timestep of each file type

In [ ]:
import cfgrib

run_dir = RUNS['baserun']

print('=== ICMGGac3u+000012 (surface fields) ===')
gg_file = os.path.join(run_dir, 'ICMGGac3u+000012')
for ds in cfgrib.open_datasets(gg_file, backend_kwargs={'indexpath': ''}):
    print('  variables:', list(ds.data_vars))

print()
print('=== ICMSHac3u+000012 (pressure-level fields) ===')
sh_file = os.path.join(run_dir, 'ICMSHac3u+000012')
for ds in cfgrib.open_datasets(sh_file, backend_kwargs={'indexpath': ''}):
    vars_info = {v: list(ds[v].dims) for v in ds.data_vars}
    if 'isobaricInhPa' in str(list(ds.dims)):
        levels = list(ds.isobaricInhPa.values) if 'isobaricInhPa' in ds.dims else '?'
        print(f'  variables: {list(ds.data_vars)}  levels: {levels}')
    else:
        print(f'  variables: {vars_info}')

## 3. Load MSL and check spatial coverage

In [ ]:
run = OIFSRun(RUNS['baserun'])

# Load first timestep of MSL
msl_t0 = run.get_msl_timestep(0)
print('MSL at t=0:')
print(msl_t0)
print(f'  lat range: {float(msl_t0.lat.min()):.1f} to {float(msl_t0.lat.max()):.1f}')
print(f'  lon range: {float(msl_t0.lon.min()):.1f} to {float(msl_t0.lon.max()):.1f}')
print(f'  MSL range: {float(msl_t0.msl.min())/100:.1f} to {float(msl_t0.msl.max())/100:.1f} hPa')

In [ ]:
import matplotlib
matplotlib.use('Agg')  # or 'TkAgg' if display is available
import matplotlib.pyplot as plt

# Quick plot of MSL at t=0
fig, ax = plt.subplots(figsize=(12, 6))
# Subset to Atlantic region
msl_atl = msl_t0.msl.sel(
    lat=slice(5, 55), lon=slice(-100, 30)
) / 100  # Pa → hPa
c = ax.contourf(msl_atl.lon, msl_atl.lat, msl_atl,
                levels=range(990, 1030, 2), cmap='RdYlBu_r')
plt.colorbar(c, ax=ax, label='MSLP (hPa)')
ax.set_title('OIFS baserun MSL at t=0 (2024-10-03 06 UTC)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
os.makedirs('../plots', exist_ok=True)
plt.savefig('../plots/msl_t0_baserun.png', dpi=100)
plt.show()
print('Saved: ../plots/msl_t0_baserun.png')

## 4. Load geopotential and verify CPS levels (300, 600, 900 hPa)

In [ ]:
# Load geopotential at 300, 600, 900 hPa for first 5 timesteps
print('Loading geopotential at 300, 600, 900 hPa...')
# Only first 5 steps to keep it fast
import cfgrib

sh_file = os.path.join(RUNS['baserun'], 'ICMSHac3u+000012')
ds_z = cfgrib.open_dataset(
    sh_file,
    filter_by_keys={'shortName': 'z'},
    backend_kwargs={'indexpath': ''}
).rename({'latitude': 'lat', 'longitude': 'lon', 'isobaricInhPa': 'level'})

z_cps = ds_z.sel(level=[300, 600, 900])
print(z_cps)
print()

# Convert to geopotential height
G = 9.80665
zg = z_cps / G
print('Geopotential height at 300, 600, 900 hPa (m):')
for lev in [300, 600, 900]:
    zg_lev = float(zg.z.sel(level=lev).mean())
    print(f'  {lev} hPa: mean = {zg_lev:.0f} m')

## 5. Check IBTrACS availability for Kirk 2024

In [ ]:
ibtracs_path = '../data/IBTrACS.NA.v04r00.nc'

if not os.path.exists(ibtracs_path):
    print('⚠️  IBTrACS file not found.')
    print('Run the following command to download it:')
    print('  python ../scripts/download_ibtracs.py')
else:
    ds = xr.open_dataset(ibtracs_path)
    y2024 = ds.where(ds.season == 2024, drop=True)

    names = []
    for n in y2024.name.values:
        try:
            name = bytes(n).strip(b'\x00').decode()
        except Exception:
            name = str(n)
        names.append(name)

    print(f'2024 NA storms ({len(names)}): {[n for n in names if n.strip()]}')
    kirk_present = any('KIRK' in n.upper() for n in names)
    if kirk_present:
        print('✅ KIRK found in IBTrACS 2024 season')
    else:
        print('⚠️  KIRK not yet in this IBTrACS version → update the file')

## 6. Summary: Data Readiness Checklist

In [ ]:
print('=' * 60)
print('HURRICANE KIRK DATA READINESS CHECKLIST')
print('=' * 60)

for run_name, run_dir in RUNS.items():
    gg = sorted(glob.glob(os.path.join(run_dir, 'ICMGGac3u+??????')))
    sh = sorted(glob.glob(os.path.join(run_dir, 'ICMSHac3u+??????')))
    ok_gg = '✅' if gg else '❌'
    ok_sh = '✅' if sh else '❌'
    gg_days = int(gg[-1].split('+')[-1]) / 24 if gg else 0
    print(f'{run_name:10s}: GG={ok_gg} ({len(gg):3d} files, {gg_days:.0f} days)  '
          f'SH={ok_sh} ({len(sh):3d} files)')

ibt = '✅' if os.path.exists('../data/IBTrACS.NA.v04r00.nc') else '❌ (run download_ibtracs.py)'
print(f'IBTrACS       : {ibt}')

try:
    import haversine
    hav = '✅'
except ImportError:
    hav = '❌ (pip install haversine)'
print(f'haversine pkg : {hav}')

try:
    import matplotlib
    mpl = '✅'
except ImportError:
    mpl = '❌ (pip install matplotlib)'
print(f'matplotlib    : {mpl}')